In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [3]:
# Kyle's Lambda -> 1/lambda implies liquidity
def Kyle_liquidity(data):
    df = pd.DataFrame()
    df['date'] = data['timestamp'].dt.date
    df['delta_p_t'] = data['close'] - data['close'].shift(1)
    df['delta_price'] = data['close'] - data['close'].shift(1)
    df['b_t'] = np.sign(df['delta_price'])
    df['b_t'] = df['b_t'].replace(0, method='ffill')
    df['b_t'] = df['b_t'].fillna(1)
    df['signed_volume'] = df['b_t'] * data['volume']
    kyle_lambda_list = []
    grouped = df.dropna(subset=['delta_p_t', 'signed_volume']).groupby('date')
    for _, group in grouped:
        if len(group) > 1:
            X, y = group['signed_volume'], group['delta_p_t']
            X = sm.add_constant(X)
            model = sm.OLS(y, X)
            results = model.fit()
            lambda_hat = results.params['signed_volume']
            df.loc[group.index, 'Kyle_liquidity'] = lambda_hat
        else:
            df.loc[group.index, 'Kyle_liquidity'] = np.nan
    data['Kyle_liquidity'] = df['Kyle_liquidity']
    return data